# Branch-Level Demand Forecasting Engine (V3)

Forecasts demand per **(branch, P/N)** instead of collapsing to the national "ALL" rollup, using 11 methods (MA, WMA, EWMA, Linear Regression, Polynomial Regression degree 2/3, SES, DES, and the intermittent-demand family Croston / SBA / TSB) and picks the best *eligible* method per part by backtest MAE.

**Multi-agency:** the engine auto-detects an `Agc` column if one is present and carries it straight through to the output, so a single consolidated export covering several agencies (e.g. a `pmovdcE` file with an "All" sheet spanning agc 06, 07, 08, 10, ...) is forecast in one pass -- branch, agency, and P/N together form the row key, so the same P/N in the same branch under two different agencies is kept as two separate rows.

**Output naming:** the output columns are named to match `CNH_FD_Processed.ipynb` / `FD_CNH.ipynb` so this notebook's output can be fed straight into that pipeline without any renaming: the branch column is `brc` (not `branch`), and the national rollup rows (see Section 3) are labeled `brc='National'` (not `'ALL'`), with `agc='ALL AGC'` for the grand total across every agency.

**Input:** you set `INPUT_FILE`, `SHEET_NAME`, and `SKIP_ROWS` to match your export (see Section 1). Leave `SHEET_NAME`/`SKIP_ROWS` as `None` and they're auto-detected instead, so small format changes won't break it.

**Model selection:** before picking a "best" model per part, the engine classifies each (branch, P/N) row's demand pattern (Smooth / Erratic / Intermittent / Lumpy / Dead, via the Syntetos-Boylan ADI/CV² framework), detects structural level shifts and one-off bulk-order months, and restricts the candidate model set accordingly -- so a volatile or shifting part can no longer get assigned a Poly-2/Poly-3 fit it isn't suited for just because it happened to win on backtest MAE. A forecast floor/fallback safety net also catches any remaining near-zero forecast for a part that's still actively moving. See Section 3.

**New in V3 -- parameter optimization:** EWMA (`alpha`), SES (`alpha`), and DES (`alpha`, `beta`) no longer use a single fixed smoothing constant for every row. Each is fit with a per-row grid search that picks whichever `alpha`/`beta` minimizes in-sample one-step-ahead MAE for that specific (branch, P/N) -- the same idea as the existing WMA weight-grid search, just applied to the exponential-smoothing family. The chosen parameters are carried through to the output as audit columns (`ewma_alpha`, `ses_alpha`, `des_alpha`, `des_beta`).

**New in V3 -- Croston / SBA / TSB:** three models purpose-built for intermittent/lumpy demand are added and made eligible wherever `Intermittent` or `Lumpy` classification applies (previously those rows were restricted to `ma`/`wma`/`ses`, none of which are structurally correct for sparse demand). Croston's Method separates demand size and inter-arrival interval and smooths each independently; SBA (Syntetos-Boylan Approximation) applies Croston's well-known positive-bias correction; TSB (Teunter-Syntetos-Babai) replaces the interval mechanic with a smoothed demand-*probability* per period, which lets its forecast decay toward zero if a part goes obsolete instead of staying flat forever the way Croston/SBA do. See Section 2b for the models and Section 3 for eligibility.

**Performance:** every model is vectorized with numpy across all rows at once (no per-row Python loops, no per-row sklearn/statsmodels calls), which is what makes branch-level (~30k+ rows) practical to run in seconds instead of minutes -- including the new parameter grid searches, which loop only over the (small) grid of candidate alpha/beta combinations, not over rows.


In [1]:
import glob
import os
import re
import time

import numpy as np
import pandas as pd

## 1. File loading

Give it the sheet name and the number of rows to skip before the header row -- the same two things you'd set in a plain `pd.read_excel(file, sheet_name=..., skiprows=...)` call. Set either one to `None` (the default) to have it auto-detected instead, which is handy if you don't know the layout yet or it shifts slightly between exports.


In [2]:
def find_data_sheet(path):
    """Picks the sheet most likely to hold the movement data.
    Prefers a sheet whose name contains 'pmovdc'; falls back to the first sheet.
    Only used when sheet_name isn't given explicitly to load_movement_data().
    """
    xl = pd.ExcelFile(path)
    for name in xl.sheet_names:
        if 'pmovdc' in name.lower():
            return name
    return xl.sheet_names[0]


def find_header_row(path, sheet_name, max_scan=15):
    """Scans the first `max_scan` rows to find the one that looks like the
    real header row (contains a P/N column and a branch column), regardless
    of how many metadata rows precede it. Only used when skiprows isn't
    given explicitly to load_movement_data().
    """
    preview = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=max_scan)
    for i, row in preview.iterrows():
        vals = [str(v).strip().lower() for v in row.values]
        has_pn = any(v in ('p/n', 'pn', 'part number') for v in vals)
        has_branch = any(v in ('brc', 'branch', 'br') for v in vals)
        if has_pn and has_branch:
            return i
    raise ValueError(
        f"Could not auto-detect the header row in the first {max_scan} rows of "
        f"sheet '{sheet_name}'. Open the file and check where the real column "
        f"headers (P/N, Brc, etc.) start, then pass skiprows explicitly."
    )


def load_movement_data(file_path, sheet_name=None, skiprows=None):
    """Loads the file you point it at.

    sheet_name / skiprows: pass these explicitly once you know them (e.g. a
    multi-agency export like "pmovdcE_CNH_2_Jul_26.xlsx" -> sheet_name="All",
    skiprows=4) for a fast, deterministic load. Leave either as None and it's
    auto-detected instead, so the notebook still works if the export layout
    shifts slightly (extra sheets, an inserted metadata row, etc.).
    """
    sheet = sheet_name if sheet_name is not None else find_data_sheet(file_path)
    rows_to_skip = skiprows if skiprows is not None else find_header_row(file_path, sheet)
    df = pd.read_excel(file_path, sheet_name=sheet, skiprows=rows_to_skip)
    df.columns = [str(c).strip().replace(chr(10), ' ').lower() for c in df.columns]
    print(f"Loaded: {file_path}")
    print(f"  sheet: '{sheet}', skiprows: {rows_to_skip}, rows: {len(df)}")
    return df


## 2. Vectorized forecasting models

Each function takes a `(n_rows, 12)` actual-demand array and returns a `(n_rows, 13)` forecast series (12 in-sample fitted points + 1 forward forecast), computed for **all rows at once**.

In [3]:
def batch_ma(clipped_15):
    """Simple moving average, 3-point window, over a 15-length input -> 13 outputs."""
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(clipped_15, window_shape=3, axis=1)  # (n,13,3)
    return windows.mean(axis=2)


def batch_wma(clipped_15, d_last, step=0.05):
    """Weighted moving average with weight-grid search (w3 > w2 > w1, sum=1).
    Loops only over the ~100-150 valid weight combos, not over rows.
    """
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(clipped_15, window_shape=3, axis=1)
    n = clipped_15.shape[0]

    combos = []
    for w1 in np.round(np.arange(0.15, 0.81, step), 2):
        for w2 in np.round(np.arange(0.25, 0.86 - w1, step), 2):
            w3 = round(1 - (w1 + w2), 2)
            if w3 > w2 > w1:
                combos.append((w1, w2, w3))

    best_rmse = np.full(n, np.inf)
    best_series = np.zeros((n, 13))
    best_weights = np.zeros((n, 3))

    for w1, w2, w3 in combos:
        forecast = (windows[:, :, 0] * w1 + windows[:, :, 1] * w2 + windows[:, :, 2] * w3) / (w1 + w2 + w3)
        rmse = np.abs(d_last - forecast[:, -1])
        better = rmse < best_rmse
        best_rmse = np.where(better, rmse, best_rmse)
        best_series[better] = forecast[better]
        best_weights[better] = [w1, w2, w3]

    return best_series, best_weights

**Fix applied (this version):** `batch_ewma` previously compared its own
in-formula value against the actual it had just used to compute that value --
a target-leakage bug, not a normal overfitting risk. That's why `batch_ewma_opt`
was landing on alpha ~0.95 for a large share of rows: it was rewarded for
leaking, not for forecasting skill. Fixed by lagging the fitted array by one
step, matching the convention `batch_ses`/`batch_des` already used correctly.


In [ ]:
def batch_ewma(clipped_12, alpha=0.4):
    """Single fixed-alpha EWMA with a genuine one-step-ahead fitted array.

    FIX (was a target-leakage bug): the original formula computed
    ewma[:, t] = alpha*x[t] + (1-alpha)*ewma[:, t-1] and then compared
    ewma[:, t] directly against x[t] when scoring alpha candidates. Since
    x[t] is baked directly into ewma[:, t]'s own formula, that comparison
    trivially improves as alpha -> 1 (at alpha=1, fitted[t] = x[t] exactly,
    MAE = 0) regardless of any real forecasting skill -- which is exactly
    why batch_ewma_opt kept landing on alpha ~= 0.95 across huge numbers of
    rows. This version keeps the same recursion for the internal smoothing
    *state* (level), but the array compared against actuals is now shifted
    by one step, so fitted[:, t] only ever uses state built through t-1 --
    identical convention to batch_ses/batch_des below, which never had
    this bug (their fitted arrays were already correctly lagged).
    """
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    for t in range(1, T):
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * level[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = clipped_12[:, 0]          # no prior state for the first point
    fitted[:, 1:T] = level[:, 0:T - 1]       # predict t using state through t-1 only
    fitted[:, T] = level[:, T - 1]           # forward forecast -- genuinely out-of-sample
    return fitted


def batch_ewma_opt(clipped_12, alpha_grid=np.round(np.arange(0.05, 0.96, 0.05), 2)):
    """Per-row alpha grid search. Same pattern as batch_wma's weight search:
    loop over the (small) grid of alpha candidates -- not over rows -- and
    keep whichever alpha gives the lowest one-step-ahead MAE for each row.
    Index 0 is excluded from the MAE since fitted[:,0] has no prior state
    to have predicted from (same treatment as batch_ses_opt/batch_des_opt).
    Returns (best_series (n,13), best_alpha (n,)).
    """
    n, T = clipped_12.shape
    best_mae = np.full(n, np.inf)
    best_series = np.zeros((n, T + 1))
    best_alpha = np.zeros(n)
    for alpha in alpha_grid:
        series = batch_ewma(clipped_12, alpha)
        fitted = series[:, 1:T]
        mae = np.abs(clipped_12[:, 1:] - fitted).mean(axis=1)
        better = mae < best_mae
        best_mae = np.where(better, mae, best_mae)
        best_series[better] = series[better]
        best_alpha[better] = alpha
    return best_series, best_alpha


In [5]:
def _batch_poly(clipped_12, degree):
    """Closed-form OLS fit, vectorized across all rows at once.
    The x-grid (1..12) is identical for every row, so a single matrix solve
    replaces thousands of individual sklearn .fit() calls.
    """
    n, T = clipped_12.shape
    x = np.arange(1, T + 1, dtype=float)
    X = np.column_stack([x ** p for p in range(degree + 1)])          # (T, deg+1)
    XtX_inv = np.linalg.pinv(X.T @ X)
    Y = clipped_12.T                                                   # (T, n)
    beta = XtX_inv @ X.T @ Y                                           # (deg+1, n)

    x_full = np.arange(1, T + 2, dtype=float)                          # 13 points
    X_full = np.column_stack([x_full ** p for p in range(degree + 1)])
    pred = X_full @ beta                                                # (13, n)
    return pred.T


def batch_lr(clipped_12):
    return _batch_poly(clipped_12, degree=1)


def batch_pr2(clipped_12):
    return _batch_poly(clipped_12, degree=2)


def batch_pr3(clipped_12):
    return _batch_poly(clipped_12, degree=3)

In [ ]:
def batch_ses(clipped_12, alpha=0.8):
    """Single fixed-alpha SES. Kept for reference -- the pipeline uses
    batch_ses_opt() below by default."""
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    for t in range(1, T):
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * level[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = clipped_12[:, 0]
    fitted[:, 1:T] = level[:, 0:T - 1]
    fitted[:, T] = level[:, T - 1]
    return fitted


def batch_ses_opt(clipped_12, alpha_grid=np.round(np.arange(0.05, 0.96, 0.05), 2)):
    """Per-row alpha grid search for SES. fitted[:,1:T] are genuine
    one-step-ahead forecasts (unlike EWMA's in-sample fit), so those are
    exactly what's compared against actuals to score each candidate alpha.
    Returns (best_series (n,13), best_alpha (n,)).
    """
    n, T = clipped_12.shape
    best_mae = np.full(n, np.inf)
    best_series = np.zeros((n, T + 1))
    best_alpha = np.zeros(n)
    for alpha in alpha_grid:
        series = batch_ses(clipped_12, alpha)
        fitted = series[:, 1:T]
        mae = np.abs(clipped_12[:, 1:] - fitted).mean(axis=1)
        better = mae < best_mae
        best_mae = np.where(better, mae, best_mae)
        best_series[better] = series[better]
        best_alpha[better] = alpha
    return best_series, best_alpha


def batch_des(clipped_12, alpha=0.1, beta=0.1):
    """Single fixed-alpha/beta DES (Holt's linear method). Kept for
    reference -- the pipeline uses batch_des_opt() below by default."""
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    trend = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    trend[:, 0] = clipped_12[:, 1] - clipped_12[:, 0]
    for t in range(1, T):
        prev_pred = level[:, t - 1] + trend[:, t - 1]
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * prev_pred
        trend[:, t] = beta * (level[:, t] - level[:, t - 1]) + (1 - beta) * trend[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = level[:, 0]
    for t in range(1, T):
        fitted[:, t] = level[:, t - 1] + trend[:, t - 1]
    fitted[:, T] = level[:, T - 1] + trend[:, T - 1]
    return fitted


def batch_des_opt(clipped_12,
                   alpha_grid=np.round(np.arange(0.1, 0.91, 0.1), 2),
                   beta_grid=np.round(np.arange(0.1, 0.91, 0.1), 2)):
    """Per-row (alpha, beta) grid search for DES. Coarser 0.1 step than the
    single-parameter models since this is a 2-D grid (9x9 = 81 combos) --
    still trivial at branch-level row counts since the search loops over
    combos, never over rows. The first two points (level0/trend0 warm-up)
    are excluded from the MAE used to score each combo.
    Returns (best_series (n,13), best_alpha (n,), best_beta (n,)).
    """
    n, T = clipped_12.shape
    best_mae = np.full(n, np.inf)
    best_series = np.zeros((n, T + 1))
    best_alpha = np.zeros(n)
    best_beta = np.zeros(n)
    for alpha in alpha_grid:
        for beta in beta_grid:
            series = batch_des(clipped_12, alpha, beta)
            fitted = series[:, 2:T]
            mae = np.abs(clipped_12[:, 2:] - fitted).mean(axis=1)
            better = mae < best_mae
            best_mae = np.where(better, mae, best_mae)
            best_series[better] = series[better]
            best_alpha[better] = alpha
            best_beta[better] = beta
    return best_series, best_alpha, best_beta


## 2b. Intermittent-demand models: Croston, SBA, TSB (new)

MA/WMA/EWMA/SES/DES all produce a forecast that tracks *every* period's demand. That's the
wrong shape of model for a part that sells in occasional bursts with long zero-demand gaps --
smoothing a mostly-zero series just drags the forecast toward zero between orders.

The Croston family instead separates two questions: *when a sale happens, how big is it?* and
*how often does a sale happen?* -- and smooths each one separately, only updating on the periods
where a sale actually occurred:

- **Croston** -- smooths demand *size* (`Z`) and the *inter-arrival interval* (`P`) independently
  with the same `alpha`; forecast = `Z / P`. Between non-zero periods the forecast stays flat
  (no new information has arrived).
- **SBA (Syntetos-Boylan Approximation)** -- Croston is a ratio of two smoothed estimates, which
  is a biased estimator (it runs high on average); SBA multiplies Croston's forecast by
  `(1 - alpha/2)` to correct for that.
- **TSB (Teunter-Syntetos-Babai)** -- replaces the interval mechanic with a smoothed
  *probability of demand this period* (`p`), updated **every** period (not just on occurrence),
  multiplied by a smoothed demand size (`z`). This is the one model of the three whose forecast
  can decay toward zero if a part stops moving -- Croston/SBA's forecast stays flat forever after
  the last sale, which is the wrong behavior for an obsoleted part.

All three get their own per-row parameter grid search (`alpha` for Croston/SBA, `alpha_p`/`alpha_z`
for TSB), same pattern as the EWMA/SES/DES optimizers in Section 2.


In [ ]:
def batch_croston(clipped_12, alpha=0.1):
    """Croston's Method. Returns a (n, 13) array: 12 in-sample one-step-ahead
    fitted values + 1 forward forecast. Z (demand size level) and P (inter-
    arrival interval level) are only updated on periods where demand > 0;
    the forecast (Z/P) is carried forward unchanged on zero-demand periods.
    """
    n, T = clipped_12.shape
    d = clipped_12
    Z = np.zeros(n)
    P = np.ones(n)
    q = np.ones(n)                      # periods since the last non-zero demand
    initialized = np.zeros(n, dtype=bool)
    fitted = np.zeros((n, T))

    for t in range(T):
        # forecast for period t is whatever state was carried in from before t
        fitted[:, t] = np.where(initialized, Z / np.maximum(P, 1e-6), 0.0)

        occurred = d[:, t] > 0
        first_occurrence = occurred & ~initialized
        Z[first_occurrence] = d[first_occurrence, t]
        P[first_occurrence] = q[first_occurrence]
        initialized = initialized | first_occurrence

        subsequent = occurred & initialized & ~first_occurrence
        Z[subsequent] = alpha * d[subsequent, t] + (1 - alpha) * Z[subsequent]
        P[subsequent] = alpha * q[subsequent] + (1 - alpha) * P[subsequent]

        q = np.where(occurred, 1, q + 1)

    forward = np.where(initialized, Z / np.maximum(P, 1e-6), 0.0)
    return np.concatenate([fitted, forward[:, None]], axis=1)


def batch_croston_opt(clipped_12, alpha_grid=np.round(np.arange(0.05, 0.51, 0.05), 2)):
    """Per-row alpha grid search for Croston. Grid is conventionally kept
    <= 0.5 for Croston/SBA -- larger alphas make the size/interval estimates
    too reactive to a single large or small order.
    Returns (best_series (n,13), best_alpha (n,)).
    """
    n, T = clipped_12.shape
    best_mae = np.full(n, np.inf)
    best_series = np.zeros((n, T + 1))
    best_alpha = np.zeros(n)
    for alpha in alpha_grid:
        series = batch_croston(clipped_12, alpha)
        mae = np.abs(clipped_12 - series[:, :T]).mean(axis=1)
        better = mae < best_mae
        best_mae = np.where(better, mae, best_mae)
        best_series[better] = series[better]
        best_alpha[better] = alpha
    return best_series, best_alpha


def batch_sba(clipped_12, alpha=0.1):
    """SBA = Croston's forecast, debiased by (1 - alpha/2)."""
    return batch_croston(clipped_12, alpha) * (1 - alpha / 2)


def batch_sba_opt(clipped_12, alpha_grid=np.round(np.arange(0.05, 0.51, 0.05), 2)):
    """Per-row alpha grid search for SBA (scored on the debiased series --
    the optimal alpha for SBA isn't necessarily the same as for raw Croston).
    Returns (best_series (n,13), best_alpha (n,)).
    """
    n, T = clipped_12.shape
    best_mae = np.full(n, np.inf)
    best_series = np.zeros((n, T + 1))
    best_alpha = np.zeros(n)
    for alpha in alpha_grid:
        series = batch_sba(clipped_12, alpha)
        mae = np.abs(clipped_12 - series[:, :T]).mean(axis=1)
        better = mae < best_mae
        best_mae = np.where(better, mae, best_mae)
        best_series[better] = series[better]
        best_alpha[better] = alpha
    return best_series, best_alpha


def batch_tsb(clipped_12, alpha_p=0.1, alpha_z=0.1):
    """TSB. p (demand-probability level) is smoothed every period; z (demand-
    size level) only updates on periods where demand > 0. Forecast = p * z.
    Returns a (n, 13) array: 12 in-sample one-step-ahead fitted values + 1
    forward forecast.
    """
    n, T = clipped_12.shape
    d = clipped_12
    p = np.zeros(n)
    z = np.zeros(n)
    fitted = np.zeros((n, T))
    for t in range(T):
        fitted[:, t] = p * z
        occurred = d[:, t] > 0
        p = alpha_p * occurred.astype(float) + (1 - alpha_p) * p
        z = np.where(occurred, alpha_z * d[:, t] + (1 - alpha_z) * z, z)
    forward = p * z
    return np.concatenate([fitted, forward[:, None]], axis=1)


def batch_tsb_opt(clipped_12,
                   alpha_p_grid=np.round(np.arange(0.05, 0.31, 0.05), 2),
                   alpha_z_grid=np.round(np.arange(0.1, 0.91, 0.1), 2)):
    """Per-row (alpha_p, alpha_z) grid search for TSB. Index 0 excluded from
    the MAE (p=z=0 before any data is seen, so fitted[:,0] is always 0).

    alpha_p_grid is capped at 0.05-0.3 (vs. alpha_z's full 0.1-0.9): alpha_p
    smooths the demand-*probability* signal, which is a noisy binary series
    over just 12 points -- letting it swing up to 0.9 makes p chase
    individual occurrence/non-occurrence flips rather than track the
    underlying probability, which is what the forecasting literature on
    TSB generally recommends against. alpha_z (the demand-size smoothing,
    only updated on non-zero periods) doesn't have the same failure mode
    and keeps the full grid.

    NOTE: this in-sample grid search (fit alpha on the same 12 points it's
    scored against) still has no held-out split -- capping the grid bounds
    how far it can swing but doesn't remove the underlying small-sample
    selection risk. A rolling-origin split (fit on months 1-9, score on
    10-12) would be the fuller fix if this becomes worth the added
    complexity.
    Returns (best_series (n,13), best_alpha_p (n,), best_alpha_z (n,)).
    """
    n, T = clipped_12.shape
    best_mae = np.full(n, np.inf)
    best_series = np.zeros((n, T + 1))
    best_ap = np.zeros(n)
    best_az = np.zeros(n)
    for ap in alpha_p_grid:
        for az in alpha_z_grid:
            series = batch_tsb(clipped_12, ap, az)
            fitted = series[:, 1:T]
            mae = np.abs(clipped_12[:, 1:] - fitted).mean(axis=1)
            better = mae < best_mae
            best_mae = np.where(better, mae, best_mae)
            best_series[better] = series[better]
            best_ap[better] = ap
            best_az[better] = az
    return best_series, best_ap, best_az


## 3. Demand pattern classification & model eligibility

Picking the "best" model purely by backtest MAE can hand a volatile or level-shifting part a
Poly-2/Poly-3 fit that happens to score well on the backtest window but collapses toward zero
(or an unrealistic curve) on the forward forecast -- mathematically defensible, operationally
wrong. It can just as easily hand an intermittent/lumpy part a model that was never designed for
mostly-zero series in the first place.

This section adds, per `(branch, P/N)` row, fully vectorized across all rows at once:

1. **ADI / CV&sup2; classification** (Syntetos-Boylan convention: CV&sup2; computed on
   *non-zero* demand months only) into `Smooth`, `Erratic`, `Intermittent`, `Lumpy`, or `Dead`.
2. **Level-shift detection** -- splits the trailing 12 months into two 6-month halves and flags
   a structural shift when one half's average is 2x (or more) the other's.
3. **Bulk-order outlier detection** -- uses the `C-1..C-12` call-count columns to flag months
   where units-per-call is a robust (median/MAD) outlier vs. that row's own history, i.e. a
   likely one-off bulk PO rather than repeat demand.
4. **Model eligibility matrix** -- restricts the candidate set fed into best-model selection
   based on (1) and (2): trend/curve models (`lr`, `pr2`, `pr3`) are dropped whenever a level
   shift is detected, and further restricted by classification. **New:** `Intermittent` and
   `Lumpy` rows are now eligible for `croston`, `sba`, and `tsb` -- the models actually built for
   that demand shape -- in addition to the existing `ma`/`wma`/`ses` fallbacks.
5. **Forecast floor / fallback** -- a last-line safety net applied after best-model selection:
   if the chosen model's forward forecast falls below 40% of the trailing 6-month average (and
   that average isn't itself ~0), it's overridden with a recency-weighted fallback instead of
   shipping a near-zero number for a part that's still moving.


In [ ]:
"""
Vectorized demand-pattern classification, level-shift detection, bulk-order
outlier flagging, model-eligibility restriction, and forecast floor/fallback.
All operations run across every row at once (no per-row Python loops), so
this scales to branch-level files with tens of thousands of rows.
"""
import numpy as np

MODEL_NAMES = ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des', 'croston', 'sba', 'tsb']

# Model eligibility per SBC classification (before level-shift adjustment)
ELIGIBLE_BY_CLASS = {
    'Smooth':       ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des'],
    'Erratic':      ['wma', 'ewma', 'ses', 'des', 'lr'],
    'Intermittent': ['ma', 'wma', 'ses', 'croston', 'sba', 'tsb'],
    'Lumpy':        ['ses', 'croston', 'sba', 'tsb'],
    'Dead':         ['ma', 'ses'],
}
TREND_MODELS = ['lr', 'pr2', 'pr3']  # excluded on level shift regardless of class


def classify_demand_pattern(d12, adi_cutoff=1.32, cv2_cutoff=0.49):
    """
    d12: (n, 12) array of the most recent 12 months of demand.
    Returns adi, cv2, classification (all length-n arrays) using the
    standard Syntetos-Boylan convention: CV^2 on non-zero demand sizes only,
    ADI = periods / count of non-zero periods.
    """
    n, T = d12.shape
    count_nz = np.sum(d12 > 0, axis=1)
    count_nz_safe = np.maximum(count_nz, 1)

    adi = T / count_nz_safe
    adi = np.where(count_nz == 0, np.inf, adi)

    sum_nz = np.sum(np.where(d12 > 0, d12, 0), axis=1)
    sumsq_nz = np.sum(np.where(d12 > 0, d12 ** 2, 0), axis=1)
    mean_nz = sum_nz / count_nz_safe
    var_nz = np.maximum(sumsq_nz / count_nz_safe - mean_nz ** 2, 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        cv2 = np.where(mean_nz > 0, var_nz / (mean_nz ** 2), 0.0)
    cv2 = np.where(count_nz < 2, 0.0, cv2)  # can't estimate variance from <2 points

    classification = np.where(
        count_nz == 0, 'Dead',
        np.where(
            (adi < adi_cutoff) & (cv2 < cv2_cutoff), 'Smooth',
            np.where(
                (adi >= adi_cutoff) & (cv2 < cv2_cutoff), 'Intermittent',
                np.where((adi < adi_cutoff) & (cv2 >= cv2_cutoff), 'Erratic', 'Lumpy')
            )
        )
    )
    return adi, cv2, classification


def detect_level_shift(d12, ratio_threshold=2.0):
    """Splits the 12-month window into two 6-month halves and flags a
    structural level shift when one half's average is >= ratio_threshold x
    the other's."""
    first_half = d12[:, :6]
    second_half = d12[:, 6:]
    avg1 = first_half.mean(axis=1)
    avg2 = second_half.mean(axis=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(avg1 > 0, avg2 / np.where(avg1 == 0, np.nan, avg1), np.inf)
    ratio = np.where((avg1 == 0) & (avg2 == 0), 1.0, ratio)

    shifted = (ratio >= ratio_threshold) | ((ratio <= 1 / ratio_threshold) & (avg1 > 0))
    shifted = np.nan_to_num(shifted.astype(float), nan=0.0).astype(bool)
    direction = np.where(avg2 > avg1, 'increasing', np.where(avg2 < avg1, 'decreasing', 'flat'))
    return avg1, avg2, ratio, shifted, direction


def detect_bulk_outliers(d12, c12, z_thresh=1.5):
    """d12/c12: (n,12) demand and calls for the same 12-month window.
    Flags months where units-per-call is a robust (median/MAD) outlier
    vs. that same row's own history -- signals a one-off bulk PO."""
    with np.errstate(divide='ignore', invalid='ignore'):
        upc = np.where(c12 > 0, d12 / np.where(c12 == 0, 1, c12), np.nan)

    median = np.nanmedian(upc, axis=1, keepdims=True)
    mad = np.nanmedian(np.abs(upc - median), axis=1, keepdims=True)
    mad_safe = np.where(mad == 0, 1e-6, mad)
    robust_z = 0.6745 * (upc - median) / mad_safe

    outlier_mask = np.nan_to_num(robust_z, nan=-np.inf) >= z_thresh
    has_outlier = np.any(outlier_mask, axis=1)
    outlier_month_count = np.sum(outlier_mask, axis=1)
    return has_outlier, outlier_month_count


def build_eligibility_matrix(classification, level_shift):
    """Returns (n, len(MODEL_NAMES)) boolean matrix: True = model eligible for that row."""
    n = len(classification)
    elig = np.zeros((n, len(MODEL_NAMES)), dtype=bool)
    for cls, models in ELIGIBLE_BY_CLASS.items():
        rows = classification == cls
        for m in models:
            elig[rows, MODEL_NAMES.index(m)] = True
    # level shift: strip trend/curve models regardless of class
    for m in TREND_MODELS:
        idx = MODEL_NAMES.index(m)
        elig[level_shift, idx] = False
    # safety: never leave a row with zero eligible models
    no_elig = ~elig.any(axis=1)
    if no_elig.any():
        elig[no_elig, MODEL_NAMES.index('ses')] = True
    return elig


def select_best_eligible_model(mae_matrix, eligibility):
    """mae_matrix, eligibility: (n, len(MODEL_NAMES)). Masks ineligible models
    to +inf before taking the argmin, so the chosen model always comes from
    the eligible set."""
    masked = np.where(eligibility, mae_matrix, np.inf)
    best_idx = np.nanargmin(masked, axis=1)
    return np.array(MODEL_NAMES)[best_idx], best_idx


def apply_forecast_floor(forecast_value, d_last6, floor_pct=0.4):
    """d_last6: (n,6) trailing 6 months of actual demand.
    If forecast falls below floor_pct * trailing 6mo avg (and that avg is
    > 0), override with a recency-weighted (WMA-style) fallback instead of
    shipping a near-zero forecast for a part that's still moving."""
    trailing_avg = d_last6.mean(axis=1)
    floor_value = floor_pct * trailing_avg

    weights = np.arange(1, d_last6.shape[1] + 1, dtype=float)
    wma_fallback = np.average(d_last6, axis=1, weights=weights)

    triggered = (forecast_value < floor_value) & (trailing_avg > 0)
    final = np.where(triggered, wma_fallback, forecast_value)
    return final, triggered, floor_value


if __name__ == "__main__":
    # quick sanity check vs ABC123
    demand = np.array([[2, 0, 2, 1, 0, 5, 7, 10, 9, 6, 2, 7]], dtype=float)
    calls = np.array([[1, 0, 1, 1, 0, 2, 3, 8, 5, 1, 1, 5]], dtype=float)
    adi, cv2, cls = classify_demand_pattern(demand)
    avg1, avg2, ratio, shifted, direction = detect_level_shift(demand)
    has_out, out_count = detect_bulk_outliers(demand, calls)
    elig = build_eligibility_matrix(cls, shifted)
    print("ADI", adi, "CV2", cv2, "class", cls)
    print("shift ratio", ratio, "shifted", shifted, direction)
    print("outlier", has_out, out_count)
    print("eligible models:", np.array(MODEL_NAMES)[elig[0]])

    # lumpy row -- should now include croston/sba/tsb
    lumpy = np.array([[0, 0, 5, 0, 0, 0, 8, 0, 0, 3, 0, 0]], dtype=float)
    adi2, cv22, cls2 = classify_demand_pattern(lumpy)
    elig2 = build_eligibility_matrix(cls2, np.array([False]))
    print("lumpy class:", cls2, "eligible models:", np.array(MODEL_NAMES)[elig2[0]])


## 4. Metrics, alerts, and the main pipeline

**A note on MAE for Croston/SBA/TSB vs. the other models:** `compute_metrics()` below applies
the *exact same* MAE formula (`mean(|actual - fitted|)` over the 12-month backtest window) to
every model, Croston/SBA/TSB included -- there's no special-cased metric per model family, so
the numbers are computed identically and are directly comparable in that mechanical sense.

What's worth knowing is that MAE is a less *informative* yardstick for intermittent/lumpy rows
than for smooth ones, for two reasons: (1) most periods in an intermittent series have actual
demand = 0, so a model that just forecasts near-zero every period can post a deceptively low MAE
even though it never predicts the actual sale months -- Croston/SBA/TSB are deliberately biased
*away* from that trivial solution, which can occasionally cost them the MAE comparison against
a model that's really just underforecasting; and (2) Croston/SBA's forecast is intentionally flat
between sales (see Section 2b), so its error is concentrated entirely in the sale months rather
than spread evenly like MA/EWMA/SES's error tends to be -- same MAE formula, structurally
different error distribution behind it. This is exactly the kind of thing MASE (MAE scaled
against a naive-forecast baseline) is generally preferred for on intermittent demand, since it
normalizes for scale and the zero-heavy series instead of comparing raw MAE across model
families directly -- worth adding as a future refinement if Croston/SBA/TSB end up winning
selection on a lot of rows and you want more confidence in *why*.


In [ ]:
MODEL_NAMES = ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des', 'croston', 'sba', 'tsb']

# models whose optimized parameter(s) we surface as audit columns on the FD window
OPTIMIZED_PARAM_MODELS = {
    'ewma': ('ewma_alpha',),
    'ses': ('ses_alpha',),
    'des': ('des_alpha', 'des_beta'),
    'croston': ('croston_alpha',),
    'sba': ('sba_alpha',),
    'tsb': ('tsb_alpha_p', 'tsb_alpha_z'),
}


def run_all_models(d_matrix_16, ub, use_fd_window):
    """Fit all 11 models on either the backtest window or the FD window.
    EWMA/SES/DES/Croston/SBA/TSB use their per-row-optimized parameter grid
    search; the chosen parameters are returned alongside the series so the
    caller can log them as audit columns.
    """
    if use_fd_window:
        actual_12 = np.clip(d_matrix_16[:, -12:], 0, ub[:, None])          # last 12 months
        clipped_15 = np.clip(d_matrix_16[:, -15:], 0, ub[:, None])         # last 15 months
    else:
        actual_12 = np.clip(d_matrix_16[:, -13:-1], 0, ub[:, None])        # 12 months before last
        clipped_15 = np.clip(d_matrix_16[:, :15], 0, ub[:, None])          # first 15 months

    d_last = d_matrix_16[:, -1]

    ewma_series, ewma_alpha = batch_ewma_opt(actual_12)
    ses_series, ses_alpha = batch_ses_opt(actual_12)
    des_series, des_alpha, des_beta = batch_des_opt(actual_12)
    croston_series, croston_alpha = batch_croston_opt(actual_12)
    sba_series, sba_alpha = batch_sba_opt(actual_12)
    tsb_series, tsb_alpha_p, tsb_alpha_z = batch_tsb_opt(actual_12)

    series = {
        'ma': batch_ma(clipped_15),
        'wma': batch_wma(clipped_15, d_last)[0],
        'ewma': ewma_series,
        'lr': batch_lr(actual_12),
        'pr2': batch_pr2(actual_12),
        'pr3': batch_pr3(actual_12),
        'ses': ses_series,
        'des': des_series,
        'croston': croston_series,
        'sba': sba_series,
        'tsb': tsb_series,
    }
    params = {
        'ewma_alpha': ewma_alpha,
        'ses_alpha': ses_alpha,
        'des_alpha': des_alpha,
        'des_beta': des_beta,
        'croston_alpha': croston_alpha,
        'sba_alpha': sba_alpha,
        'tsb_alpha_p': tsb_alpha_p,
        'tsb_alpha_z': tsb_alpha_z,
    }
    return series, actual_12, params


def compute_metrics(actual_12, series):
    """Vectorized RMSE / MAE / R2 / MAPE / SMAPE / MASE per row per model.
    Applies the identical formula to every model in MODEL_NAMES.

    MASE (Mean Absolute Scaled Error) = MAE / naive_mae, where naive_mae is
    the row's own in-sample one-step naive (persistence) MAE: mean(|actual[t]
    - actual[t-1]|) across the same 12-month window. This gives a
    scale-independent, cross-model-family-comparable metric: MASE < 1 means
    "beats naive", MASE > 1 means "worse than just guessing last period's
    value". This is what makes Croston/SBA/TSB (whose raw MAE tends to run
    structurally lower than MA/WMA/SES simply because they forecast mostly-
    zero series) comparable to the other models on genuinely equal footing --
    see build_eligibility_matrix / the selection_matrix logic in forecast()
    for where this gets used as the Intermittent/Lumpy selection criterion.
    """
    metrics = {}
    mean_actual = actual_12.mean(axis=1, keepdims=True)
    ss_tot = ((actual_12 - mean_actual) ** 2).sum(axis=1)
    ss_tot_safe = np.where(ss_tot == 0, np.nan, ss_tot)

    naive_mae = np.abs(np.diff(actual_12, axis=1)).mean(axis=1)
    naive_mae_safe = np.where(naive_mae == 0, np.nan, naive_mae)

    for name in MODEL_NAMES:
        pred = series[name][:, :12]
        err = actual_12 - pred
        rmse = np.sqrt((err ** 2).mean(axis=1))
        mae = np.abs(err).mean(axis=1)
        ss_res = (err ** 2).sum(axis=1)
        r2 = 1 - ss_res / ss_tot_safe
        with np.errstate(divide='ignore', invalid='ignore'):
            mape = np.nanmean(np.where(actual_12 == 0, np.nan, np.abs(err) / np.abs(actual_12)), axis=1) * 100
            smape = np.mean(np.abs(err) / (np.abs(actual_12) + np.abs(pred) + 1e-10), axis=1) * 100
            mase = mae / naive_mae_safe
        metrics[name] = dict(RMSE=rmse, MAE=mae, R2=r2, MAPE=mape, SMAPE=smape, MASE=mase)
    return metrics


def compute_alerts(d_matrix_16):
    last12 = d_matrix_16[:, -12:]
    mean_a = last12.mean(axis=1)
    median_a = np.median(last12, axis=1)
    std_a = last12.std(axis=1)
    max_a = last12.max(axis=1)

    spike = (max_a > np.maximum(mean_a * 1.8, median_a * 1.8)).astype(int)
    with np.errstate(divide='ignore', invalid='ignore'):
        cv = np.where(mean_a != 0, std_a / mean_a, 0)
    volatility = (cv > 1).astype(int)
    zero_count = (last12 == 0).sum(axis=1)
    intermittent = (zero_count >= 6).astype(int)

    score = spike + volatility + intermittent
    label = np.where(score >= 2, 'HIGH RISK', np.where(score == 1, 'REVIEW', 'OK'))
    return spike, volatility, intermittent, score, label


In [ ]:
def find_demand_columns(df):
    cols = [c for c in df.columns if re.match(r'^d-\d+$', c)]
    if not cols:
        raise ValueError(
            "Could not find demand columns matching pattern 'd-<number>' "
            "(e.g. d-1, d-2, ... d-16). Check your column headers."
        )
    return sorted(cols, key=lambda x: int(x.split('-')[1]), reverse=True)


def find_calls_columns(df):
    """Finds C-1..C-12 call-count columns. Optional -- if absent, bulk-order
    outlier detection is skipped and every row's bulk_outlier_flag is False."""
    cols = [c for c in df.columns if re.match(r'^c-\d+$', c)]
    return sorted(cols, key=lambda x: int(x.split('-')[1]), reverse=True)


def find_branch_column(df):
    for candidate in ('brc', 'branch', 'br'):
        if candidate in df.columns:
            return candidate
    raise ValueError("Could not find a branch column (expected 'brc' or 'branch').")


def find_pn_column(df):
    for candidate in ('p/n', 'pn', 'part number'):
        if candidate in df.columns:
            return candidate
    raise ValueError("Could not find a P/N column.")


def find_agc_column(df):
    for candidate in ('agc', 'agency'):
        if candidate in df.columns:
            return candidate
    return None


def forecast(df, include_national_rollup=False,
             adi_cutoff=1.32, cv2_cutoff=0.49,
             level_shift_ratio=2.0, outlier_z=1.5, floor_pct=0.4):
    """Main entry point. Auto-detects branch/P/N/agc/demand/calls columns.

    Output columns match CNH_FD_Processed.ipynb / FD_CNH.ipynb's naming:
    'brc' (not 'branch'). When include_national_rollup=True and an agc
    column is present, two extra tiers of rows are added on top of the raw
    (brc, agc, p/n) rows:
      - brc='National', agc=<real agency>   -> that agency's total across all branches
      - brc='National', agc='ALL AGC'       -> the grand total across every branch
                                                AND every agency, per P/N

    Section 3: each row is classified by demand pattern (ADI/CV2), checked
    for a level shift and bulk-order outliers, and best-model selection is
    restricted to the resulting eligible model set (now including Croston /
    SBA / TSB for Intermittent/Lumpy rows). A forecast floor/fallback is
    applied to the final forward forecast.

    EWMA/SES/DES/Croston/SBA/TSB all use per-row-optimized parameters (see
    Section 2 / 2b); the chosen alpha/beta values are logged as audit
    columns (ewma_alpha, ses_alpha, des_alpha, des_beta, croston_alpha,
    sba_alpha, tsb_alpha_p, tsb_alpha_z).

    adi_cutoff / cv2_cutoff: Syntetos-Boylan classification thresholds.
    level_shift_ratio: half-over-half average ratio that counts as a shift.
    outlier_z: robust z-score threshold for flagging a bulk-order month.
    floor_pct: forecast floor as a fraction of the trailing 6-month average.
    """
    df = df.copy()
    df.columns = [str(c).strip().replace(chr(10), ' ').lower() for c in df.columns]

    branch_col = find_branch_column(df)
    pn_col = find_pn_column(df)
    agc_col = find_agc_column(df)
    demand_cols = find_demand_columns(df)
    calls_cols = find_calls_columns(df)

    df[pn_col] = df[pn_col].astype(str).str.upper().str.strip()

    keep = [branch_col, pn_col] + ([agc_col] if agc_col else []) + demand_cols + calls_cols
    branch_df = df[keep].rename(columns={branch_col: 'brc', pn_col: 'p/n'})
    if agc_col:
        branch_df = branch_df.rename(columns={agc_col: 'agc'})
    else:
        branch_df['agc'] = np.nan

    frames = [branch_df]
    if include_national_rollup:
        if agc_col:
            # Tier 1: national total PER agency -- sums every branch, keeps agc separate
            national_per_agc = branch_df.groupby(['agc', 'p/n'], as_index=False)[demand_cols + calls_cols].sum()
            national_per_agc.insert(0, 'brc', 'National')
            frames.append(national_per_agc[['brc', 'agc', 'p/n'] + demand_cols + calls_cols])

            # Tier 2: grand total across every branch AND every agency for each P/N
            national_all_agc = branch_df.groupby(['p/n'], as_index=False)[demand_cols + calls_cols].sum()
            national_all_agc.insert(0, 'agc', 'ALL AGC')
            national_all_agc.insert(0, 'brc', 'National')
            frames.append(national_all_agc[['brc', 'agc', 'p/n'] + demand_cols + calls_cols])
        else:
            national = branch_df.groupby(['p/n'], as_index=False)[demand_cols + calls_cols].sum()
            national.insert(0, 'brc', 'National')
            frames.append(national[['brc', 'p/n'] + demand_cols + calls_cols])

    out = pd.concat(frames, ignore_index=True)
    d_matrix = out[demand_cols].to_numpy(dtype=float)                                # (n, 16)
    c_matrix = out[calls_cols].to_numpy(dtype=float) if calls_cols else None          # (n, 12)
    n = len(out)

    ub_backtest = d_matrix[:, -13:-1].mean(axis=1) + 1.5 * d_matrix[:, -13:-1].std(axis=1)
    ub_fd = d_matrix[:, -12:].mean(axis=1) + 1.5 * d_matrix[:, -12:].std(axis=1)

    series_bt, actual_bt, _params_bt = run_all_models(d_matrix, ub_backtest, use_fd_window=False)
    metrics_bt = compute_metrics(actual_bt, series_bt)
    mae_matrix = np.column_stack([metrics_bt[m]['MAE'] for m in MODEL_NAMES])
    mase_matrix = np.column_stack([metrics_bt[m]['MASE'] for m in MODEL_NAMES])

    series_fd, actual_fd, params_fd = run_all_models(d_matrix, ub_fd, use_fd_window=True)
    metrics_fd = compute_metrics(actual_fd, series_fd)

    # ---- demand pattern classification (Section 3), on the FD 12-month window ----
    d12_fd = d_matrix[:, -12:]
    adi, cv2, classification = classify_demand_pattern(d12_fd, adi_cutoff, cv2_cutoff)
    avg1, avg2, shift_ratio, level_shifted, shift_direction = detect_level_shift(d12_fd, level_shift_ratio)

    if c_matrix is not None:
        has_outlier, outlier_month_count = detect_bulk_outliers(d12_fd, c_matrix, outlier_z)
    else:
        has_outlier = np.zeros(n, dtype=bool)
        outlier_month_count = np.zeros(n, dtype=int)

    eligibility = build_eligibility_matrix(classification, level_shifted)

    # ---- selection metric: MAE everywhere, EXCEPT Intermittent/Lumpy rows use
    # MASE instead. Smooth/Erratic (and Dead) never compare across the
    # Croston/SBA/TSB family, so MAE stays perfectly comparable there and is
    # left untouched. Intermittent/Lumpy rows DO compare MA/WMA/SES against
    # Croston/SBA/TSB, whose raw MAE runs structurally lower simply because
    # they forecast mostly-zero series -- MASE (scaled against each row's own
    # naive one-step error) puts every model on the same footing for that
    # comparison. Rows where MASE is undefined (naive_mae == 0) fall back to
    # MAE rather than being dropped from selection.
    use_mase = np.isin(classification, ['Intermittent', 'Lumpy'])
    selection_matrix = mae_matrix.copy()
    mase_rows = mase_matrix[use_mase]
    mae_fallback_rows = mae_matrix[use_mase]
    mase_rows = np.where(np.isnan(mase_rows), mae_fallback_rows, mase_rows)
    selection_matrix[use_mase] = mase_rows

    best_model, best_idx = select_best_eligible_model(selection_matrix, eligibility)
    selection_metric_used = np.where(use_mase, 'MASE', 'MAE')

    result = pd.DataFrame({
        'brc': out['brc'],
        'agc': out['agc'],
        'p/n': out['p/n'],
    })
    result['clipped_d_FD'] = list(actual_fd)
    for name in MODEL_NAMES:
        result[f'{name}_FD'] = list(series_fd[name])

    result['best_model'] = best_model

    fd_forecast_raw = np.array([series_fd[best_model[i]][i, -1] for i in range(n)])

    # ---- forecast floor / fallback safety net ----
    # Croston/SBA/TSB are intermittent-demand-native models: a low or even
    # zero forward forecast from them can be the CORRECT signal (e.g. a part
    # that only shows demand every few months, forecast during an off month),
    # not a degenerate collapse like Poly3 hitting zero on volatile data
    # (the original motivating case). Applying the same floor built for
    # MA/WMA/EWMA/SES/DES would systematically overstate exactly the rows
    # these three models exist to handle correctly. Exempt them entirely.
    INTERMITTENT_NATIVE_MODELS = ('croston', 'sba', 'tsb')
    floor_exempt = np.isin(best_model, INTERMITTENT_NATIVE_MODELS)

    d_last6 = d_matrix[:, -6:]
    fd_forecast_floored, floor_triggered_raw, floor_value = apply_forecast_floor(
        fd_forecast_raw, d_last6, floor_pct
    )
    fd_forecast_final = np.where(floor_exempt, fd_forecast_raw, fd_forecast_floored)
    floor_triggered = np.where(floor_exempt, False, floor_triggered_raw)

    result['FD_forecast_raw'] = fd_forecast_raw
    result['FD_forecast'] = fd_forecast_final
    result['FD_final'] = np.maximum(0, np.round(fd_forecast_final)).astype(int)

    result['metrics_FD'] = [
        [
            {'model': f'{m}_FD', **{k: metrics_fd[m][k][i] for k in ['RMSE', 'MAE', 'R2', 'MAPE', 'SMAPE']}}
            for m in MODEL_NAMES
        ]
        for i in range(n)
    ]
    best_r2_fd = np.array([metrics_fd[best_model[i]]['R2'][i] for i in range(n)])
    best_mase_fd = np.array([metrics_fd[best_model[i]]['MASE'][i] for i in range(n)])
    result['best_r2_FD'] = best_r2_fd
    result['best_mase_FD'] = best_mase_fd
    # BUG FIX: NaN < 0.25 evaluates False in numpy, so rows with undefined R2
    # (constant 12-month actual -- almost always Dead/all-zero parts) were
    # silently falling into the 'Good' bucket -- the opposite of useful,
    # since R2 is meaningless for those rows in the first place.
    result['r2_status_FD'] = np.where(
        np.isnan(best_r2_fd), 'N/A (no variance)',
        np.where(best_r2_fd < 0.25, 'R2 < 0.25', 'Good')
    )
    result['selection_metric_used'] = selection_metric_used

    spike, volatility, intermittent, score, label = compute_alerts(d_matrix)
    result['spike_alert'] = spike
    result['volatility_alert'] = volatility
    result['intermittent_alert'] = intermittent
    result['forecast_alert_score'] = score
    result['forecast_alert_label'] = label

    # ---- classification + model-selection audit trail columns ----
    result['adi'] = np.round(np.where(np.isinf(adi), 999.0, adi), 2)
    result['cv2'] = np.round(cv2, 3)
    result['demand_classification'] = classification
    result['level_shift_detected'] = level_shifted
    result['level_shift_ratio'] = np.round(np.where(np.isinf(shift_ratio), 999.0, shift_ratio), 2)
    result['level_shift_direction'] = shift_direction
    result['bulk_outlier_flag'] = has_outlier
    result['bulk_outlier_months'] = outlier_month_count
    result['eligible_models'] = ['|'.join(np.array(MODEL_NAMES)[eligibility[i]]) for i in range(n)]
    result['excluded_models'] = ['|'.join(np.array(MODEL_NAMES)[~eligibility[i]]) for i in range(n)]
    result['forecast_floor_triggered'] = floor_triggered
    result['forecast_floor_value'] = np.round(floor_value, 2)

    # ---- optimized-parameter audit columns (new in V3) ----
    for pname, pvals in params_fd.items():
        result[pname] = np.round(pvals, 3)

    return result


## 5. Run it

Set `INPUT_FILE` to your current `pmovdcE` export each time you run this. `SHEET_NAME` and `SKIP_ROWS` below are set for a multi-agency CNH export (`pmovdcE_CNH_2_Jul_26.xlsx`, sheet "All", header on row 5, i.e. `skiprows=4`) -- change them to match whatever file you're using, or set both to `None` to auto-detect instead.


In [10]:
INPUT_FILE = "pmovdcE CNH Non Steron 19Aug26.xlsx"   # <- change this to your current export
SHEET_NAME = "Non Steron"                          # e.g. "All", "CNH Steron", "CNH Non Steron" -- or None to auto-detect
SKIP_ROWS = 4                               # rows before the header row -- or None to auto-detect
INCLUDE_NATIONAL_ROLLUP = True   # set False if you only want branch-level rows

t0 = time.time()
raw = load_movement_data(INPUT_FILE, sheet_name=SHEET_NAME, skiprows=SKIP_ROWS)

t1 = time.time()
result = forecast(raw, include_national_rollup=INCLUDE_NATIONAL_ROLLUP)
print(f"Forecast computed for {len(result)} rows in {time.time()-t1:.1f}s")
if result['agc'].notna().any():
    # sort key as str() since agc can mix real agency codes (int) with the
    # 'ALL AGC' grand-total label (str) once national rollup is included
    agencies = sorted(result['agc'].dropna().unique().tolist(), key=str)
    print(f"Agencies in this file ({len(agencies)}): {agencies}")


Loaded: pmovdcE CNH Non Steron 19Aug26.xlsx
  sheet: 'Non Steron', skiprows: 4, rows: 40318


C:\Users\Brandon\AppData\Local\Temp\ipykernel_28524\146209268.py:43: RuntimeWarning: Mean of empty slice
  mape = np.nanmean(np.where(actual_12 == 0, np.nan, np.abs(err) / np.abs(actual_12)), axis=1) * 100
C:\Users\Brandon\AppData\Local\Temp\ipykernel_28524\1581907240.py:83: RuntimeWarning: All-NaN slice encountered
  median = np.nanmedian(upc, axis=1, keepdims=True)
C:\Users\Brandon\AppData\Local\Temp\ipykernel_28524\1581907240.py:84: RuntimeWarning: All-NaN slice encountered
  mad = np.nanmedian(np.abs(upc - median), axis=1, keepdims=True)


Forecast computed for 69334 rows in 2.9s
Agencies in this file (10): [10, 15, 21, 26, 33, 38, 6, 7, 8, 'ALL AGC']


In [11]:
print("Demand classification breakdown:")
print(result['demand_classification'].value_counts())
print()
print("Level shift detected:")
print(result['level_shift_detected'].value_counts())
print()
print("Bulk-order outlier flagged:")
print(result['bulk_outlier_flag'].value_counts())
print()
print("Forecast floor triggered:")
print(result['forecast_floor_triggered'].value_counts())
print()
print("Best model distribution:")
print(result['best_model'].value_counts())


Demand classification breakdown:
demand_classification
Dead            36408
Intermittent    30793
Lumpy            1195
Smooth            707
Erratic           231
Name: count, dtype: int64

Level shift detected:
level_shift_detected
False    41425
True     27909
Name: count, dtype: int64

Bulk-order outlier flagged:
bulk_outlier_flag
False    65663
True      3671
Name: count, dtype: int64

Forecast floor triggered:
forecast_floor_triggered
False    59936
True      9398
Name: count, dtype: int64

Best model distribution:
best_model
ma      44847
ses     23068
ewma      895
wma       481
pr3        39
pr2         4
Name: count, dtype: int64


In [12]:
os.makedirs("output", exist_ok=True)
filename = f"output/forecast_branch_{time.strftime('%Y-%m-%d')}.xlsx"
result.to_excel(filename, index=False)
print(f"Saved to {filename}  ({os.path.getsize(filename)/1e6:.1f} MB)")
print(f"Total runtime: {time.time()-t0:.1f}s")

result.head()

Saved to output/forecast_branch_2026-08-19.xlsx  (16.3 MB)
Total runtime: 67.6s


,brc,agc,p/n,clipped_d_FD,ma_FD,wma_FD,ewma_FD,lr_FD,pr2_FD,pr3_FD,...,demand_classification,level_shift_detected,level_shift_ratio,level_shift_direction,bulk_outlier_flag,bulk_outlier_months,eligible_models,excluded_models,forecast_floor_triggered,forecast_floor_value
0,20,6,47364317,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,Dead,False,1.0,flat,False,0,ma|ses,wma|ewma|lr|pr2|pr3|des,False,0.0
1,20,6,47486870,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,Dead,False,1.0,flat,False,0,ma|ses,wma|ewma|lr|pr2|pr3|des,False,0.0
2,20,6,48017998,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,Dead,False,1.0,flat,False,0,ma|ses,wma|ewma|lr|pr2|pr3|des,False,0.0
3,20,6,48167835,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,Dead,False,1.0,flat,False,0,ma|ses,wma|ewma|lr|pr2|pr3|des,False,0.0
4,20,6,48174869,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,Dead,False,1.0,flat,False,0,ma|ses,wma|ewma|lr|pr2|pr3|des,False,0.0
